# 05 Labeling

Notebook ini membuat skor dan label impulsive spending dari feature engineering. Jika `df_features` dari notebook 04 belum ada di memory, notebook memakai fallback minimum feature dari CSV cleaned.

Output utama di memory:
- `df_labeled`: dataframe transaksi dengan score, label, dan driver.
- `label_summary_df`: distribusi label utama.
- `dataset_label_summary_df`: ringkasan label per dataset.
- `driver_summary_df`: ringkasan label berdasarkan driver utama.


## 1. Import Library


In [65]:
from pathlib import Path
import warnings
import os
from google.colab import files
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


## 2. Konfigurasi Path dan Tampilan


In [66]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

cleaned_separate_path = project_root / 'data' / 'interim' / 'cleaned_separate'

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

print(f'Project root         : {project_root}')
print(f'Cleaned separate path: {cleaned_separate_path}')
print('Mode                 : display-only. Tidak ada file output yang disimpan.')


Project root         : /content
Cleaned separate path: /content/data/interim/cleaned_separate
Mode                 : display-only. Tidak ada file output yang disimpan.


## 3. Bobot Score dan Threshold Label


In [67]:
W_NIGHT = 0.25
W_CATEGORY = 0.25
W_AMOUNT = 0.30
W_WEEKEND = 0.20
LABEL_IMPULSIF = 0.90
LABEL_PERTIMBANGAN = 0.70
LABEL_ORDER = ['AMAN', 'PERTIMBANGAN', 'IMPULSIF']


## 4. Konfigurasi Kategori


In [68]:
STANDARD_CATEGORIES = ['Makanan', 'Transportasi', 'Hiburan', 'Belanja', 'Pendidikan', 'Kesehatan', 'Tagihan', 'Lainnya']
HEDONIC_CATEGORIES = {'Hiburan', 'Belanja'}
NEUTRAL_CATEGORIES = {'Makanan', 'Transportasi', 'Lainnya'}
UTIL_CATEGORIES = {'Pendidikan', 'Kesehatan', 'Tagihan'}

DIRECT_CATEGORY_MAP = {
    'food': 'Makanan',
    'food_and_drink': 'Makanan',
    'food_drink': 'Makanan',
    'grocery': 'Makanan',
    'groceries': 'Makanan',
    'restaurant': 'Makanan',
    'snacks': 'Makanan',
    'makanan': 'Makanan',
    'transportation': 'Transportasi',
    'transport': 'Transportasi',
    'travel': 'Transportasi',
    'commute': 'Transportasi',
    'train': 'Transportasi',
    'bus': 'Transportasi',
    'transportasi': 'Transportasi',
    'entertainment': 'Hiburan',
    'subscription': 'Hiburan',
    'movies': 'Hiburan',
    'gaming': 'Hiburan',
    'culture': 'Hiburan',
    'festivals': 'Hiburan',
    'tourism': 'Hiburan',
    'social_life': 'Hiburan',
    'hiburan': 'Hiburan',
    'shopping': 'Belanja',
    'apparel': 'Belanja',
    'clothing': 'Belanja',
    'beauty': 'Belanja',
    'grooming': 'Belanja',
    'household': 'Belanja',
    'gift': 'Belanja',
    'belanja': 'Belanja',
    'education': 'Pendidikan',
    'self_development': 'Pendidikan',
    'pendidikan': 'Pendidikan',
    'health': 'Kesehatan',
    'health_and_fitness': 'Kesehatan',
    'fitness': 'Kesehatan',
    'medical': 'Kesehatan',
    'kesehatan': 'Kesehatan',
    'utilities': 'Tagihan',
    'rent': 'Tagihan',
    'bills': 'Tagihan',
    'mobile_service_provider': 'Tagihan',
    'water': 'Tagihan',
    'internet': 'Tagihan',
    'electricity': 'Tagihan',
    'phone': 'Tagihan',
    'tagihan': 'Tagihan',
    'other': 'Lainnya',
    'unknown': 'Lainnya',
    'lainnya': 'Lainnya',
}

KEYWORD_CATEGORY_RULES = [
    ('Makanan', ['makanan', 'minuman', 'food', 'snack', 'grocery', 'restaurant']),
    ('Transportasi', ['transport', 'commute', 'travel', 'train', 'bus', 'ojek']),
    ('Hiburan', ['entertainment', 'subscription', 'movie', 'gaming', 'mainan', 'festival', 'netflix', 'culture']),
    (
        'Belanja',
        [
            'shopping', 'fashion', 'pakaian', 'apparel', 'beauty', 'aksesoris', 'plastik', 'wadah', 'rak',
            'celengan', 'nampan', 'tray', 'baskom', 'mangkok', 'lunch_box', 'rantang', 'saringan',
            'pintu', 'perkakas', 'seal', 'baut', 'roof', 'household',
        ],
    ),
    ('Pendidikan', ['education', 'school', 'course', 'book', 'buku']),
    ('Kesehatan', ['health', 'fitness', 'medical', 'obat', 'olahraga']),
    ('Tagihan', ['utilities', 'rent', 'bill', 'mobile', 'water', 'internet', 'electricity', 'phone', 'listrik', 'pulsa']),
]


## 5. Category Helper


In [69]:
def clean_key(value):
    text = str(value).strip().lower().replace('&', ' and ')
    text = ''.join(character if character.isalnum() else '_' for character in text)

    while '__' in text:
        text = text.replace('__', '_')

    return text.strip('_')


def map_category(value, domain=None):
    key = clean_key(value)

    if key in DIRECT_CATEGORY_MAP:
        return DIRECT_CATEGORY_MAP[key]

    for category, keywords in KEYWORD_CATEGORY_RULES:
        if any(keyword in key for keyword in keywords):
            return category

    return 'Belanja' if domain == 'ecommerce_sales' else 'Lainnya'


## 6. Fallback Feature Helper


In [70]:
def load_cleaned_from_csv(folder):
    frames = []

    for path in sorted(folder.glob('*_cleaned.csv')):
        frame = pd.read_csv(path, low_memory=False)
        if 'timestamp' not in frame.columns and 'date' in frame.columns:
            frame['timestamp'] = frame['date']
        if 'dataset_id' not in frame.columns:
            frame['dataset_id'] = path.name.replace('_cleaned.csv', '')
        frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def prepare_cleaned_input(frame):
    data = frame.copy()
    required_columns = ['timestamp', 'amount', 'category', 'dataset_id']
    missing_columns = [column for column in required_columns if column not in data.columns]

    if missing_columns:
        raise ValueError(f'Kolom cleaned wajib belum ada: {missing_columns}')

    data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
    data['amount'] = pd.to_numeric(data['amount'], errors='coerce')
    data = data.dropna(subset=required_columns).query('amount > 0').copy()
    data['category'] = data.apply(lambda row: map_category(row['category'], row.get('domain')), axis=1)
    data['source'] = data['source'] if 'source' in data.columns else data['dataset_id']
    return data.sort_values('timestamp').reset_index(drop=True)


def resolve_cleaned_input(cleaned_frame=None):
    if isinstance(cleaned_frame, pd.DataFrame) and not cleaned_frame.empty:
        return prepare_cleaned_input(cleaned_frame), 'memory_from_03'

    return prepare_cleaned_input(load_cleaned_from_csv(cleaned_separate_path)), 'read_only_csv_fallback'


def build_input_summary(frame):
    return (
        frame.groupby('dataset_id')
        .agg(rows=('amount', 'size'), total_amount=('amount', 'sum'), median_amount=('amount', 'median'))
        .reset_index()
    )


In [71]:
def robust_z_score(series):
    values = pd.to_numeric(series, errors='coerce')
    median = values.median()
    mad = (values - median).abs().median()

    if pd.isna(mad) or mad == 0:
        std = values.std(ddof=0)
        if pd.isna(std) or std == 0:
            return pd.Series(0.0, index=series.index)
        return (values - values.mean()) / (std + 1e-9)

    return 0.6745 * (values - median) / (mad + 1e-9)


def build_minimum_features(frame):
    data = prepare_cleaned_input(frame)
    data['hour'] = data['timestamp'].dt.hour
    data['day_of_week'] = data['timestamp'].dt.dayofweek
    data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)
    data['night_score'] = (1 - (np.minimum(data['hour'], 24 - data['hour']) / 12)).clip(0, 1)
    data['time_segment'] = pd.cut(
        data['hour'],
        bins=[-1, 5, 10, 15, 19, 23],
        labels=['late_night', 'morning', 'midday', 'evening', 'night'],
    ).astype('string')

    score_map = {category: 0.0 for category in STANDARD_CATEGORIES}
    score_map.update({category: 1.0 for category in HEDONIC_CATEGORIES})
    score_map.update({category: 0.5 for category in NEUTRAL_CATEGORIES})
    data['category_score'] = data['category'].map(score_map).fillna(0)
    data['amount_z'] = data.groupby('dataset_id', group_keys=False)['amount'].transform(robust_z_score).clip(-3, 3)
    data['amount_score'] = ((data['amount_z'] + 3) / 6).clip(0, 1)
    return data.reset_index(drop=True)


## 7. Load Feature Data Helper


In [72]:
REQUIRED_LABEL_COLUMNS = [
    'timestamp',
    'dataset_id',
    'amount',
    'category',
    'night_score',
    'category_score',
    'amount_score',
    'is_weekend',
]


def resolve_label_input(feature_frame=None):
    if isinstance(feature_frame, pd.DataFrame) and not feature_frame.empty:
        return feature_frame.copy(), 'memory_from_04'

    cleaned = load_cleaned_from_csv(cleaned_separate_path)
    return build_minimum_features(cleaned), 'read_only_csv_fallback'


def validate_label_input(frame):
    missing_columns = [column for column in REQUIRED_LABEL_COLUMNS if column not in frame.columns]
    if missing_columns:
        raise ValueError(f'Kolom wajib belum ada: {missing_columns}')
    if frame.empty:
        raise ValueError('Input labeling kosong.')


## 8. Load Feature Data


In [73]:
df_cleaned = pd.read_csv(
    'df_03_new_feature_merged.csv',
    low_memory=False
)

df_cleaned = df_cleaned.rename(columns={
    'tanggal_transaksi': 'timestamp',
    'amount_idr': 'amount',
    'kategori_clean': 'category',
    'sumber_dataset': 'dataset_id'
})

df_input, input_mode = resolve_cleaned_input(df_cleaned)

print(f'Input mode : {input_mode}')

display(build_input_summary(df_input))
display(df_input.head())


Input mode : memory_from_03


,dataset_id,rows,total_amount,median_amount
0,daily_household,2452,1.237548e+09,18300.000
1,ecommerce_sales,4545,2.531376e+08,23523.000
2,personal_finance,1500,1.961281e+06,1156.285


,dataset_id,timestamp,tahun,bulan,tahun_bulan,tipe_transaksi,category,amount,metode_pembayaran,status_transaksi,sumber_file,source,hour,day_of_week,day_name,day_of_month,month,date_only,is_weekend,week_period,hour_sin,hour_cos,is_night,is_late_night,night_score,time_segment,category_score,category_type,is_hedonic_category,amount_log,amount_z,amount_score,amount_percentile,is_high_amount,user_proxy,transactions_same_day,daily_amount_total,share_of_daily_spend,amount_vs_weekly_avg,budget_limit,spent_so_far,budget_remaining_ratio,weekend_score,fingo_impulse_signal,signal_band
0,daily_household,2015-01-01,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,7.512618,-0.714176,0.380971,0.037520,0,daily_household_unknown_unknown,11,174216.0,0.010504,0.002797,1.261774e+07,1830.0,0.999855,0.0,0.595243,watch
1,daily_household,2015-01-01,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,11.200964,2.380588,0.896765,0.694535,0,daily_household_unknown_unknown,11,174216.0,0.420168,0.111884,1.261774e+07,75030.0,0.994054,0.0,0.724191,high
2,daily_household,2015-01-01,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.205492,-0.634824,0.394196,0.116232,0,daily_household_unknown_unknown,11,174216.0,0.021008,0.005594,1.261774e+07,78690.0,0.993764,0.0,0.598549,watch
3,daily_household,2015-01-01,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,9.303922,-0.317412,0.447098,0.379894,0,daily_household_unknown_unknown,11,174216.0,0.063025,0.016783,1.261774e+07,89670.0,0.992893,0.0,0.611775,watch
4,daily_household,2015-01-01,2015,1,2015-01,Expense,Lainnya,7320.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.898502,-0.476118,0.420647,0.296900,0,daily_household_unknown_unknown,11,174216.0,0.042017,0.011188,1.261774e+07,96990.0,0.992313,0.0,0.605162,watch


## 9. Score Function


In [74]:

def add_impulsive_score(frame):
    data = frame.copy()
    data['impulsive_score'] = (
        W_NIGHT * data['night_score']
        + W_CATEGORY * data['category_score']
        + W_AMOUNT * data['amount_score']
        + W_WEEKEND * data['is_weekend']
    ).mul(100).clip(0, 100)
    return data

## 10. Label Function


In [75]:
def add_impulsive_label(frame):
    data = frame.copy()
    q_mid = data['impulsive_score'].quantile(LABEL_PERTIMBANGAN)
    q_high = data['impulsive_score'].quantile(LABEL_IMPULSIF)
    data['label'] = np.select(
        [data['impulsive_score'] >= q_high, data['impulsive_score'] >= q_mid],
        ['IMPULSIF', 'PERTIMBANGAN'],
        default='AMAN',
    )
    data['is_impulsive'] = (data['label'] == 'IMPULSIF').astype(int)
    data.attrs['q_mid'] = float(q_mid)
    data.attrs['q_high'] = float(q_high)
    return data


## 11. Driver Function


In [76]:
def add_label_drivers(frame):
    data = frame.copy()
    data['driver_night'] = data['night_score'] >= 0.65
    data['driver_hedonic'] = data['category_score'] >= 1
    data['driver_high_amount'] = data['amount_score'] >= data['amount_score'].quantile(0.80)
    data['driver_weekend'] = data['is_weekend'] == 1
    data['driver_count'] = data[['driver_night', 'driver_hedonic', 'driver_high_amount', 'driver_weekend']].sum(axis=1)
    data['main_driver'] = np.select(
        [data['driver_hedonic'], data['driver_night'], data['driver_high_amount'], data['driver_weekend']],
        ['hedonic_category', 'night_time', 'high_amount', 'weekend'],
        default='mixed_low_signal',
    )
    return data


## 12. Run Labeling Helper


In [77]:
def build_label_summary(frame):
    summary = frame['label'].value_counts().reindex(LABEL_ORDER, fill_value=0).rename_axis('label').reset_index(name='rows')
    summary['share_pct'] = (summary['rows'] / len(frame) * 100).round(2)
    return summary


def run_labeling(frame):
    scored = add_impulsive_score(frame)
    labeled = add_impulsive_label(scored)
    labeled = add_label_drivers(labeled)
    return labeled, build_label_summary(labeled)


## 13. Run Labeling


In [78]:
df_label_input = pd.read_csv('df_03_new_feature_merged.csv')
df_labeled, label_summary_df = run_labeling(df_label_input)


print(f'Total rows: {len(df_labeled):,}')
print(f"Q70: {df_labeled.attrs['q_mid']:.2f}")
print(f"Q90: {df_labeled.attrs['q_high']:.2f}")
display(label_summary_df)
display(df_labeled[['timestamp', 'dataset_id', 'amount', 'category', 'impulsive_score', 'label', 'main_driver']].head())


Total rows: 8,497
Q70: 62.50
Q90: 74.32


,label,rows,share_pct
0,AMAN,5946,69.98
1,PERTIMBANGAN,1701,20.02
2,IMPULSIF,850,10.00


,timestamp,dataset_id,amount,category,impulsive_score,label,main_driver
0,2015-01-01 00:00:00,daily_household,1830.0,Transportasi,48.929118,AMAN,night_time
1,2015-01-01 00:00:00,daily_household,73200.0,Makanan,64.402941,PERTIMBANGAN,night_time
2,2015-01-01 00:00:00,daily_household,3660.0,Transportasi,49.325882,AMAN,night_time
3,2015-01-01 00:00:00,daily_household,10980.0,Transportasi,50.912941,AMAN,night_time
4,2015-01-01 00:00:00,daily_household,7320.0,Lainnya,50.119412,AMAN,night_time


In [79]:
check_features = [
    'is_night',
    'is_weekend',
    'amount_vs_weekly_avg',
    'budget_remaining_ratio'
]

behavior_check = (
    df_labeled
    .groupby('label')[check_features]
    .mean()
    .round(3)
    .reset_index()
)

display(behavior_check)

,label,is_night,is_weekend,amount_vs_weekly_avg,budget_remaining_ratio
0,AMAN,0.085,0.069,0.607,0.237
1,IMPULSIF,0.268,0.941,1.332,0.156
2,PERTIMBANGAN,0.118,0.471,1.173,0.263


## 14. Validate Label Helper


In [80]:
def build_label_validation(frame):
    return pd.DataFrame(
        [
            {
                'rows': len(frame),
                'missing_score': int(frame['impulsive_score'].isna().sum()),
                'missing_label': int(frame['label'].isna().sum()),
                'label_count': int(frame['label'].nunique()),
                'impulsive_rate_pct': round(frame['is_impulsive'].mean() * 100, 2),
            }
        ]
    )


def validate_label_output(validation):
    if validation[['missing_score', 'missing_label']].sum(axis=1).iloc[0] > 0:
        raise AssertionError('Masih ada score/label kosong')
    print('Validasi labeling lolos')


## 15. Validate Label


In [81]:
validation_df = build_label_validation(df_labeled)
display(validation_df)
validate_label_output(validation_df)


,rows,missing_score,missing_label,label_count,impulsive_rate_pct
0,8497,0,0,3,10.0


Validasi labeling lolos


## 16. Label Summary Helper


In [82]:
def build_dataset_label_summary(frame):
    return (
        frame.groupby('dataset_id')
        .agg(
            rows=('label', 'size'),
            impulsive_share=('is_impulsive', lambda value: value.mean() * 100),
            avg_score=('impulsive_score', 'mean'),
        )
        .reset_index()
    )


def build_driver_summary(frame):
    return (
        frame.groupby('main_driver')
        .agg(
            rows=('label', 'size'),
            impulsive_share=('is_impulsive', lambda value: value.mean() * 100),
            avg_score=('impulsive_score', 'mean'),
        )
        .reset_index()
    )


## 17. Label Summary


In [83]:
dataset_label_summary_df = build_dataset_label_summary(df_labeled)
driver_summary_df = build_driver_summary(df_labeled)

display(dataset_label_summary_df)
display(driver_summary_df)


,dataset_id,rows,impulsive_share,avg_score
0,daily_household,2452,8.564437,54.725003
1,ecommerce_sales,4545,9.768977,54.621428
2,personal_finance,1500,13.066667,60.072262


,main_driver,rows,impulsive_share,avg_score
0,hedonic_category,4838,10.913601,55.541138
1,high_amount,207,9.661836,55.972987
2,mixed_low_signal,501,0.000000,33.974899
3,night_time,2719,11.107025,59.900479
4,weekend,232,0.000000,53.289788


## 18. Visualisasi Label Helper


In [84]:
def sample_labeled_transactions(frame, max_rows=5000, random_state=42):
    return frame.sample(min(len(frame), max_rows), random_state=random_state)

def plot_label_overview(label_summary, labeled_frame, dataset_summary, driver_summary):
    label_chart = alt.Chart(label_summary).mark_bar().encode(
        x=alt.X('label:N', sort=LABEL_ORDER, title='Label'),
        y=alt.Y('rows:Q', title='Jumlah Transaksi'),
        color=alt.Color('label:N', legend=None),
        tooltip=['label', 'rows', 'share_pct'],
    ).properties(title='Distribusi Label', height=260)

    score_chart = alt.Chart(sample_labeled_transactions(labeled_frame)).mark_bar().encode(
        x=alt.X('impulsive_score:Q', bin=alt.Bin(maxbins=30), title='Impulsive Score'),
        y=alt.Y('count():Q', title='Jumlah Transaksi'),
        color=alt.Color('label:N', sort=LABEL_ORDER),
        tooltip=['label', 'count()'],
    ).interactive().properties(title='Distribusi Impulsive Score', height=260)

    dataset_chart = alt.Chart(dataset_summary).mark_bar().encode(
        x=alt.X('impulsive_share:Q', title='Impulsive Share (%)'),
        y=alt.Y('dataset_id:N', sort='-x', title='Dataset'),
        color=alt.Color('dataset_id:N', legend=None),
        tooltip=['dataset_id', 'rows', alt.Tooltip('impulsive_share:Q', format='.2f'), alt.Tooltip('avg_score:Q', format='.2f')],
    ).properties(title='Impulsive Share per Dataset', height=280)

    driver_chart = alt.Chart(driver_summary).mark_bar().encode(
        x=alt.X('impulsive_share:Q', title='Impulsive Share (%)'),
        y=alt.Y('main_driver:N', sort='-x', title='Driver'),
        color=alt.Color('main_driver:N', legend=None),
        tooltip=['main_driver', 'rows', alt.Tooltip('impulsive_share:Q', format='.2f'), alt.Tooltip('avg_score:Q', format='.2f')],
    ).properties(title='Impulsive Share per Driver', height=260)

    return (label_chart | score_chart) & (dataset_chart | driver_chart)


## 19. Visualisasi Label (Interactive Altair)


In [85]:
plot_label_overview(label_summary_df, df_labeled, dataset_label_summary_df, driver_summary_df)


alt.VConcatChart(...)

## 20. Visualisasi Segmentasi Helper


In [86]:
def build_segment_summary(frame):
    return (
        frame.groupby(['time_segment', 'category'], observed=True)
        .agg(
            rows=('label', 'size'),
            impulsive_share=('is_impulsive', lambda value: value.mean() * 100),
            avg_score=('impulsive_score', 'mean'),
        )
        .reset_index()
    )


def plot_segment_heatmap(segment_summary):
    return alt.Chart(segment_summary).mark_rect().encode(
        x=alt.X('time_segment:N', title='Segmen Waktu'),
        y=alt.Y('category:N', title='Kategori'),
        color=alt.Color('impulsive_share:Q', title='Impulsive Share (%)'),
        tooltip=['time_segment', 'category', 'rows', alt.Tooltip('impulsive_share:Q', format='.2f'), alt.Tooltip('avg_score:Q', format='.2f')],
    ).properties(title='Heatmap Impulsive Share: Waktu x Kategori', height=320)


## 21. Visualisasi Segmentasi


In [87]:
segment_df = build_segment_summary(df_labeled)
plot_segment_heatmap(segment_df)


alt.Chart(...)

### 22. Split Dataset

In [89]:
df_03_new_feature_merged['label'] = (
    df_03_new_feature_merged['signal_band']
    .map({
        'low': 'AMAN',
        'watch': 'PERTIMBANGAN',
        'high': 'IMPULSIF'
    })
)

train_df, temp_df = train_test_split(
    df_03_new_feature_merged,
    test_size=0.30,
    stratify=df_03_new_feature_merged['label'],
    random_state=42
)
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)
print(train_df.shape)
print(validation_df.shape)
print(test_df.shape)

(5947, 46)
(1275, 46)
(1275, 46)


## 22. Output Labeling


In [90]:
df_labeled.to_csv('df_labeled.csv', index=False)
label_summary_df.to_csv('label_summary_df.csv', index=False)

train_df.to_csv('train_df.csv', index=False)
validation_df.to_csv('validation_df.csv', index=False)
test_df.to_csv('test_df.csv', index=False)

files.download('df_labeled.csv')
files.download('label_summary_df.csv')
files.download('train_df.csv')
files.download('validation_df.csv')
files.download('test_df.csv')

print('File berhasil disimpan')
print('- df_labeled.csv')
print('- label_summary_df.csv')
print('- train_df.csv')
print('- validation_df.csv')
print('- test_df.csv')

print(f"Lokasi df_labeled.csv         : {os.path.abspath('df_labeled.csv')}")
print(f"Lokasi label_summary_df.csv  : {os.path.abspath('label_summary_df.csv')}")
print(f"Lokasi train_df.csv          : {os.path.abspath('train_df.csv')}")
print(f"Lokasi validation_df.csv     : {os.path.abspath('validation_df.csv')}")
print(f"Lokasi test_df.csv           : {os.path.abspath('test_df.csv')}")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File berhasil disimpan
- df_labeled.csv
- label_summary_df.csv
- train_df.csv
- validation_df.csv
- test_df.csv
Lokasi df_labeled.csv         : /content/df_labeled.csv
Lokasi label_summary_df.csv  : /content/label_summary_df.csv
Lokasi train_df.csv          : /content/train_df.csv
Lokasi validation_df.csv     : /content/validation_df.csv
Lokasi test_df.csv           : /content/test_df.csv


## Area Analisis Mandiri
Gunakan cell kosong di bawah untuk eksplorasi hasil labeling setelah semua cell utama dijalankan.

Function dan variabel yang bisa dipakai ulang:
- `resolve_label_input(df_features)`: memilih input dari memory notebook 04 atau fallback CSV cleaned.
- `validate_label_input(frame)`: mengecek kolom wajib sebelum scoring.
- `build_minimum_features(frame)`: membuat fitur minimum jika `df_features` belum tersedia.
- `add_impulsive_score(frame)`: menghitung skor impulsif 0-100.
- `add_impulsive_label(frame)`: membuat label `AMAN`, `PERTIMBANGAN`, dan `IMPULSIF` berdasarkan quantile.
- `add_label_drivers(frame)`: menambahkan driver utama penyebab label.
- `run_labeling(frame)`: menjalankan scoring, labeling, dan driver sekaligus.
- `build_label_validation(df_labeled)`: membuat ringkasan validasi label.
- `plot_label_overview(...)`: membuat visualisasi distribusi label, score, dataset, dan driver.
- `plot_segment_heatmap(segment_df)`: membuat heatmap label berdasarkan segmen waktu dan kategori.
- `df_labeled`: dataframe utama berisi score, label, dan driver.
- `label_summary_df`, `dataset_label_summary_df`, `driver_summary_df`: ringkasan hasil labeling.


In [92]:
df_labeled

,dataset_id,timestamp,tahun,bulan,tahun_bulan,tipe_transaksi,category,amount,metode_pembayaran,status_transaksi,sumber_file,source,hour,day_of_week,day_name,day_of_month,month,date_only,is_weekend,week_period,hour_sin,hour_cos,is_night,is_late_night,night_score,time_segment,category_score,category_type,is_hedonic_category,amount_log,amount_z,amount_score,amount_percentile,is_high_amount,user_proxy,transactions_same_day,daily_amount_total,share_of_daily_spend,amount_vs_weekly_avg,budget_limit,spent_so_far,budget_remaining_ratio,weekend_score,fingo_impulse_signal,signal_band,impulsive_score,label,is_impulsive,driver_night,driver_hedonic,driver_high_amount,driver_weekend,driver_count,main_driver
0,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.000000,1.000000,0,1,1.000000,late_night,0.5,neutral,0,7.512618,-0.714176,0.380971,0.037520,0,daily_household_unknown_unknown,11,174216.0,0.010504,0.002797,1.261774e+07,1830.0,0.999855,0.0,0.595243,watch,48.929118,AMAN,0,True,False,False,False,1,night_time
1,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.000000,1.000000,0,1,1.000000,late_night,0.5,neutral,0,11.200964,2.380588,0.896765,0.694535,0,daily_household_unknown_unknown,11,174216.0,0.420168,0.111884,1.261774e+07,75030.0,0.994054,0.0,0.724191,high,64.402941,PERTIMBANGAN,0,True,False,True,False,2,night_time
2,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.000000,1.000000,0,1,1.000000,late_night,0.5,neutral,0,8.205492,-0.634824,0.394196,0.116232,0,daily_household_unknown_unknown,11,174216.0,0.021008,0.005594,1.261774e+07,78690.0,0.993764,0.0,0.598549,watch,49.325882,AMAN,0,True,False,False,False,1,night_time
3,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.000000,1.000000,0,1,1.000000,late_night,0.5,neutral,0,9.303922,-0.317412,0.447098,0.379894,0,daily_household_unknown_unknown,11,174216.0,0.063025,0.016783,1.261774e+07,89670.0,0.992893,0.0,0.611775,watch,50.912941,AMAN,0,True,False,False,False,1,night_time
4,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Lainnya,7320.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.000000,1.000000,0,1,1.000000,late_night,0.5,neutral,0,8.898502,-0.476118,0.420647,0.296900,0,daily_household_unknown_unknown,11,174216.0,0.042017,0.011188,1.261774e+07,96990.0,0.992313,0.0,0.605162,watch,50.119412,AMAN,0,True,False,False,False,1,night_time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8492,ecommerce_sales,2025-11-30 21:36:00,2025,11,2025-11,Sales,Belanja,15900.0,COD (Bayar di Tempat),Telah Dikirim,ecommerce_sales_clean.csv,ecommerce_sales,21,6,Sunday,30,2025-11,2025-11-30,1,48,-0.707107,0.707107,1,0,0.750000,night,1.0,hedonic,1,9.674137,-0.529146,0.411809,0.176458,0,ecommerce_sales_unknown_unknown,38,1293072.0,0.012296,0.395997,1.392396e+06,8877610.0,0.000000,1.0,0.765452,high,76.104269,IMPULSIF,1,True,True,False,True,3,hedonic_category
8493,ecommerce_sales,2025-11-30 21:52:00,2025,11,2025-11,Sales,Belanja,14499.0,COD (Bayar di Tempat),Telah Dikirim,ecommerce_sales_clean.csv,ecommerce_sales,21,6,Sunday,30,2025-11,2025-11-30,1,48,-0.707107,0.707107,1,0,0.750000,night,1.0,hedonic,1,9.581904,-0.626396,0.395601,0.149175,0,ecommerce_sales_unknown_unknown,38,1293072.0,0.011213,0.361104,1.392396e+06,8892109.0,0.000000,1